In [1]:
import torch
from Teacher import ViTVideoEncoder, ViTVideoAutoencoder

def test_vit_slot_encoder():
    print("Testing ViTVideoEncoder...")
    
    # Configuration
    B = 2
    C, H, W = 1, 128, 128
    D = 512
    model = ViTVideoEncoder(img_size=H, embed_dim=D, num_slots=4)
    model.eval()

    # Test 1: Fixed number of frames
    print("\nTest 1: Fixed T=5")
    T1 = 5
    x1 = torch.randn(B, T1, C, H, W)
    with torch.no_grad():
        out1 = model(x1)
    print(f"Input: {x1.shape}, Output: {out1.shape}")
    assert out1.shape == (B, 4, D)

    # Test 2: Different number of frames
    print("\nTest 2: Fixed T=10")
    T2 = 10
    x2 = torch.randn(B, T2, C, H, W)
    with torch.no_grad():
        out2 = model(x2)
    print(f"Input: {x2.shape}, Output: {out2.shape}")
    assert out2.shape == (B, 4, D)

    print("\nEncoder tests passed!")

def test_vit_autoencoder():
    print("\nTesting ViTVideoAutoencoder (with SimpleTimeDecoder - Discrete)...")
    B = 2
    T = 6
    C, H, W = 1, 128, 128
    D = 512
    
    model = ViTVideoAutoencoder(img_size=H, embed_dim=D, num_slots=4)
    model.eval()
    
    x = torch.randn(B, T, C, H, W)
    t = torch.randint(0, 6, (B,)) # Random discrete time indices [0, 5]
    
    with torch.no_grad():
        recon, slots = model(x, t)
        
    print(f"Input Video: {x.shape}")
    print(f"Target Times: {t.shape}")
    print(f"Reconstructed Frame: {recon.shape}")
    print(f"Slots: {slots.shape}")
    
    assert recon.shape == (B, C, H, W)
    assert slots.shape == (B, 4, D)
    print("Autoencoder tests passed!")

In [2]:
test_vit_slot_encoder()

Testing ViTVideoEncoder...

Test 1: Fixed T=5
Input: torch.Size([2, 5, 1, 128, 128]), Output: torch.Size([2, 4, 512])

Test 2: Fixed T=10
Input: torch.Size([2, 10, 1, 128, 128]), Output: torch.Size([2, 4, 512])

Encoder tests passed!


In [3]:
test_vit_autoencoder()


Testing ViTVideoAutoencoder (with SimpleTimeDecoder - Discrete)...
Input Video: torch.Size([2, 6, 1, 128, 128])
Target Times: torch.Size([2])
Reconstructed Frame: torch.Size([2, 1, 128, 128])
Slots: torch.Size([2, 4, 512])
Autoencoder tests passed!


In [3]:
import torch
import numpy as np
from PoolingFlow import SpatialLatentFlow

def test_flow():
    print("Testing SpatialLatentFlow Module...")
    B, N, D = 4, 16, 512 # Use smaller dim for speed
    M = 4
    
    # Initialize implementation
    # Note: flow_hidden_dim is usually keeping D or 2*D.
    model = SpatialLatentFlow(input_dim=D, num_tokens=M, flow_depth=4, flow_hidden_dim=D)
    
    x = torch.randn(B, N, D)
    
    print(f"Input Shape: {x.shape}")
    
    # Forward
    z, log_det, pooled = model(x)
    
    print(f"Latent Shape: {z.shape}")
    print(f"Pooled Shape: {pooled.shape}")
    print(f"LogDet Shape: {log_det.shape}")
    
    expected_dim = M * D
    assert z.shape == (B, expected_dim)
    assert pooled.shape == (B, M, D)
    
    # Check Invertibility for the Flow part
    # Inverse takes z -> reconstructed flat pooled tokens
    recon_pooled = model.inverse(z)
    print(f"Reconstructed Pooled Shape: {recon_pooled.shape}")
    
    # Check difference between pooling output and reconstruction
    # Should be close to numerical precision
    diff = (pooled - recon_pooled).abs().max().item()
    print(f"Max reconstruction error: {diff}")
    
    if diff < 1e-4:
        print("Invertibility Test Passed!")
    else:
        print("Invertibility Test Failed (error too high)")
        
    print("-" * 20)


In [4]:
test_flow()

Testing SpatialLatentFlow Module...
Input Shape: torch.Size([4, 16, 512])
Latent Shape: torch.Size([4, 2048])
Pooled Shape: torch.Size([4, 4, 512])
LogDet Shape: torch.Size([4])
Reconstructed Pooled Shape: torch.Size([4, 4, 512])
Max reconstruction error: 0.0
Invertibility Test Passed!
--------------------


In [3]:
import torch
from Teacher import ViTTeacher
# Simple sanity check
model = ViTTeacher()
B, T = 64, 5
dummy_input = torch.randn(B, T, 1, 128, 128)
data = {'dimg': dummy_input}

ret, loss = model(data)
print(f"Input shape: {dummy_input.shape}")
print(f"Recon shape: {ret['PRED'].shape}")

print(ret['FRAME'].item())
assert ret['PRED'].shape == (B, 1, 128, 128)
print("Teacher Check passed!")

Input shape: torch.Size([64, 5, 1, 128, 128])
Recon shape: torch.Size([64, 1, 128, 128])
0
Teacher Check passed!


In [2]:
import torch
from Student import Student
# Simple sanity check
model = Student()

dummy_input = torch.randn(64, 5, 12, 114, 10)
data = {'csi': dummy_input,
        'dimg': torch.randn(64, 5, 1, 128, 128),
       'har': torch.randint(0, 29, (64,))}

ret, loss = model(data)
print(f"Token shape: {ret['TOKEN'].shape}")
print(f"HAR shape: {ret['HAR'].shape}")

assert ret['HAR'].shape == (64, 29)
print("Student Check passed!")

Token shape: torch.Size([64, 5, 8, 512])
HAR shape: torch.Size([64, 29])
Teacher Check passed!
